# 05 — Agente investidor (CRRA)

Desenvolve a classe `Investidor` (utilidade, carteira, consumo). Usa `nucleo` e `mercado` já prontos. **F5, F6, F8, F9.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np
from app import nucleo
from app.mercado import RendaFixa, RendaVariavel

## Desenvolvimento

A(s) função(ões)/classe(s) abaixo foi(ram) escrita(s) aqui e, após os testes, movida(s) para `app/agente.py`.

In [2]:
class Investidor:
    """Agente CRRA com decisão de consumo e portfólio. (F5)"""

    def __init__(self, gamma: float, beta: float, w0: float, horizonte: int) -> None:
        if gamma <= 0:
            raise ValueError("γ (aversão ao risco) deve ser > 0.")
        if not 0.0 < beta < 1.0:
            raise ValueError("β (fator de desconto) deve estar em (0, 1).")
        if w0 <= 0:
            raise ValueError("W₀ (riqueza inicial) deve ser > 0.")
        if horizonte < 1:
            raise ValueError("horizonte T deve ser ≥ 1.")
        self.gamma = float(gamma)
        self.beta = float(beta)
        self.w0 = float(w0)
        self.horizonte = int(horizonte)
        self._alpha_star: np.ndarray | None = None
        self._phi_hat: float | None = None

    # ── Utilidade CRRA (F5) ─────────────────────────────────────────────────
    def utilidade(self, c):
        """u(c) = c^(1−γ)/(1−γ) (ou ln c se γ=1). (F5)"""
        c = np.asarray(c, dtype=float)
        if np.isclose(self.gamma, 1.0):
            return np.log(c)
        return c ** (1.0 - self.gamma) / (1.0 - self.gamma)

    def utilidade_marginal(self, c):
        """u'(c) = c^(−γ). (F5)"""
        return np.asarray(c, dtype=float) ** (-self.gamma)

    # ── Decisão de portfólio e consumo (F6, F8, F9) ─────────────────────────
    def carteira_otima(self, mercado: RendaVariavel, rf: float, *,
                       n_scenarios: int = 100_000, seed: int | None = 42,
                       **opts) -> np.ndarray:
        """Carteira ótima α* via FOC G(α*)=0. (F6)

        Amostra cenários (líquidos) do mercado e converte para fatores brutos.
        **α é sempre irrestrito** (short e alavancagem permitidos, sem teto).

        ``opts`` é repassado a ``nucleo.resolver_alpha_otimo`` (``tol``,
        ``maxiter``, ``alpha0``).
        """
        r = mercado.amostrar(n_scenarios, seed=seed)
        rf_bruto = 1.0 + rf
        R = np.maximum(1.0 + r, 0.0)  # resp. limitada do ativo: preço não fica < 0
        alpha = nucleo.resolver_alpha_otimo(R, rf_bruto, self.gamma, **opts)
        self._alpha_star = alpha
        self._phi_hat = nucleo.phi_chapeu(alpha, R, rf_bruto, self.gamma)
        return alpha

    def fracoes_consumo(self) -> np.ndarray:
        """Frações de consumo θ_t = A_t^(−1/γ), t=0..T. (F8, F9)

        Requer ``carteira_otima(...)`` chamado antes (usa o Φ̂ guardado).
        """
        if self._phi_hat is None:
            raise RuntimeError(
                "chame carteira_otima(...) antes de fracoes_consumo() — "
                "θ_t depende de Φ̂, que vem da política ótima."
            )
        A = nucleo.recorrencia_A(self._phi_hat, self.beta, self.gamma, self.horizonte)
        return nucleo.fracoes_consumo(A, self.gamma)

    @property
    def alpha_star(self) -> np.ndarray | None:
        """Última carteira ótima α* calculada (ou None)."""
        return self._alpha_star

    @property
    def phi_hat(self) -> float | None:
        """Φ̂ da última política ótima (ou None)."""
        return self._phi_hat


**Teste** — utilidade CRRA e decisão ótima (carteira + consumo).

In [3]:
import pandas as pd
inv = Investidor(gamma=5.0, beta=0.96, w0=1.0, horizonte=12)
print('u(2):', inv.utilidade(2.0), "| u'(2):", inv.utilidade_marginal(2.0))
assert np.isclose(inv.utilidade(2.0), 2.0**(-4)/(-4)) and np.isclose(inv.utilidade_marginal(2.0), 2.0**(-5))  # gamma=5

u(2): -0.015625 | u'(2): 0.03125


In [4]:
rng = np.random.default_rng(7); ruido = rng.normal(0,0.06,300); ruido -= ruido.mean()
ret = pd.DataFrame({'data': pd.date_range('2000-01',periods=300,freq='MS').strftime('%Y-%m'),'ibov':0.015+ruido})
alpha = inv.carteira_otima(RendaVariavel(ret), RendaFixa(0.10).retorno_livre_risco(), n_scenarios=40_000, seed=1)
print('carteira_otima:', alpha, '| alpha_star:', inv.alpha_star, '| phi_hat:', inv.phi_hat)
theta = inv.fracoes_consumo(); print('theta:', theta)
assert np.isclose(theta[-1], 1.0) and np.all(np.diff(theta) > 0)
print('F5/F6/F8 Investidor: PASSOU')

carteira_otima: [0.43327271] | alpha_star: [0.43327271] | phi_hat: 0.9633213419184827
theta: [0.08434563 0.09068584 0.09818248 0.10718248 0.11818697 0.13194762
 0.14964566 0.17324977 0.20630361 0.25589447 0.33855939 0.50390943
 1.        ]
F5/F6/F8 Investidor: PASSOU
